<a href="https://colab.research.google.com/github/dhaev/Data-projects/blob/main/freight_analysis/freight_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import libraries

In [27]:
!pip install diskcache
import os, glob
import time
import json
import sqlite3
import hashlib
from diskcache import Cache
import requests
import math
import plotly.express as px
import pandas as pd
import numpy as np
from datetime import date, timedelta

# Pandas settings

In [28]:
pd.set_option('display.max_columns', None)       # Show all columns
pd.set_option('display.width', None)             # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)      # Show full content in each column

# API settings
previously used to calculate accurate distance between locations and get geometry. However due to rate limit, *harvesine distance* is currently used instead.

In [29]:
OPENROUTESERVICE_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjBjZjE2MDQ5YTgzOTRjNDJhYjA3NzI3OWZkODY3MDY1IiwiaCI6Im11cm11cjY0In0=" # Replace with your actual ORS API key
OPENROUTESERVICE_API_URL = "https://api.openrouteservice.org/v2/directions/driving-hgv/geojson"
ORS_ROUTE_CACHE = Cache("/content/drive/MyDrive/freight_analysis/cache/ors_routes_cache")
DRIVING_HOURS = 10
SECONDS_TO_HOURS = 3600

# Freight Database
Stores persistent information used across data processing tasks

In [30]:
# --- Database Setup ---
# Define the database file name
DB_FILE = '/content/drive/MyDrive/freight_analysis/freight.db'
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Get route geometry
Uses OpenRouteService API to get the *route*, *distance* and *duration* of a trip between two locations.

In [31]:
#Get route info
# --- Helper function to get route geometry from OpenRouteService ---
def get_route_geometry(start_lon, start_lat, end_lon, end_lat, api_key=OPENROUTESERVICE_API_KEY):
    """
    Fetches route geometry (list of [lat, lon] points) from OpenRouteService.
    Returns None if the route cannot be found or an error occurs.
    Caches results to reduce API calls using diskcache.
    """
    cache_key = (start_lon, start_lat, end_lon, end_lat)
    cached_result = ORS_ROUTE_CACHE.get(cache_key)
    if cached_result is not None: # Check for None explicitly, as a valid route could be an empty list if ORS returns no geometry
        # print(f"Fetching ORS route from disk cache for {cache_key}")
        return cached_result

    headers = {
        'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
        'Authorization': api_key,
        'Content-Type': 'application/json; charset=utf-8'
    }
    # Coordinates format for ORS is [longitude, latitude]
    body = {
        "coordinates": [[start_lon, start_lat], [end_lon, end_lat]]
    }

    try:
        time.sleep(7)
        response = requests.post(OPENROUTESERVICE_API_URL, headers=headers, json=body)#, timeout=60)
        # response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        data = response.json()
        status = response.status_code
        print(status)

        # Extract coordinates from the GeoJSON response
        if status == 200 and data and 'features' in data and len(data['features']) > 0:
          feature = data['features'][0]
          geometry = feature['geometry']['coordinates']
          summary = feature['properties']['summary']

          distance = float(summary['distance']) * 0.000621371
          duration_in_seconds = float(summary['duration'])
          duration_in_hours = duration_in_seconds / SECONDS_TO_HOURS
          duration_in_driving_days = duration_in_hours / DRIVING_HOURS

          plotly_geometry = [[point[1], point[0]] for point in geometry]

          cache_value = {
              'status': status,
              'geometry': geometry,
              'plotly_geometry': plotly_geometry,
              'distance': distance,
              'duration_in_seconds': duration_in_seconds,
              'duration_in_hours': duration_in_hours,
              'duration_in_driving_days': duration_in_driving_days
          }
          ORS_ROUTE_CACHE.set(cache_key, cache_value)
          return cache_value
        else:
            print(f"No route features found for {start_lat},{start_lon} to {end_lat},{end_lon}  [[{start_lon},{start_lat}],[{end_lon},{end_lat}] ]")
            ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result
            return {'status': status}
    except requests.exceptions.RequestException as e:
        print(f"Error fetching route from OpenRouteService: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response body: {e.response.text}")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None
    except json.JSONDecodeError:
        print(f"Error decoding JSON response from OpenRouteService for route {start_lat},{start_lon} to {end_lat},{end_lon} [[{start_lon},{start_lat}],[{end_lon},{end_lat}] ]")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None
    except Exception as e:
        print(f"An unexpected error occurred in get_route_geometry: {e}")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None

# Retrieve or create equipment types
Stores all equipment(truck types) as unique values in an sqlite database

In [32]:
# --- Helper Function for Equipment ID Management ---
def get_or_create_equipment_id(equipment_name, cursor, conn):
    """
    Checks if an equipment name exists in the 'equipments' table.
    If it exists, returns its ID. If not, inserts it and returns the new ID.
    """
    if equipment_name is None:
        return None

    # Try to find the equipment by name
    cursor.execute("SELECT id FROM equipments WHERE equipment = ?", (equipment_name,))
    result = cursor.fetchone()

    if result:
        # Equipment found, return its ID
        return result[0]
    else:
        # Equipment not found, insert it
        cursor.execute("INSERT INTO equipments (equipment) VALUES (?)", (equipment_name,))
        conn.commit() # Commit the insert operation
        # Get the ID of the last inserted row
        return cursor.lastrowid

# Data ingestion-Nextload Loadboard

In [33]:
def get_nextload_data():
  extracted_data =  []
  folder_path = '/content/drive/MyDrive/freight_analysis/loadReq'
  if os.path.exists(folder_path):
      for f in glob.glob(folder_path + '/*.json'):
          try:
              # Open and load the JSON file
              with open(str(f), 'r', encoding='utf-8') as file:
                  nextload_data = json.load(file)
          except FileNotFoundError:
              print(f"Error: The JSON file '{json_file_path}' was not found. Please ensure it's in the correct directory.")
          except json.JSONDecodeError as e:
              print(f"Error decoding JSON from '{json_file_path}': {e}")
          except sqlite3.Error as e:
              print(f"SQLite database error: {e}")
          except Exception as e:
              print(f"An unexpected error occurred: {e}")


          loads = nextload_data.get('loads', []) # Safely get the 'loads' list
          # Keep track of all unique equipment display names encountered for testing
          all_processed_equipment_names = set()

          for load in loads:
              # Extract equipment display names
              equipment_display_names = [
                  eq_type.get("displayName")
                  for eq_type in load.get("equipmentTypes", [])
                  if eq_type.get("displayName") is not None # Ensure displayName exists
              ]

              # Convert equipment display names to database IDs
              equipment_type_ids = []
              for eq_name in equipment_display_names:
                  eq_id = get_or_create_equipment_id(eq_name, cursor, conn)
                  if eq_id is not None:
                      equipment_type_ids.append(eq_id)
                      all_processed_equipment_names.add(eq_name) # Add to set for testing

              # Build the dictionary for the current load, using .get() for safe access
              extracted_data.append({
                  "source": "nextload",
                  "source_id": load.get("id"),
                  "load_reference": load.get("referenceNumber"),
                  "post_date": load.get("originalPostingDate"),
                  "last_updated": load.get("postingDate"), # Corrected key to 'postingDate' as per JSON snippet

                  "pickup_id": load.get("pick", {}).get("id", {}),
                  "pickup_city": load.get("pick", {}).get("location", {}).get("city"),
                  "pickup_state": load.get("pick", {}).get("location", {}).get("state"),
                  "pickup_country": load.get("pick", {}).get("location", {}).get("country"),
                  "pickup_latitude": load.get("pick", {}).get("location", {}).get("latitude"),
                  "pickup_longitude": load.get("pick", {}).get("location", {}).get("longitude"),
                  "pickup_date": load.get("pick", {}).get("startDate"),

                  "drop_id": load.get("drop", {}).get("id", {}),
                  "drop_city": load.get("drop", {}).get("location", {}).get("city"),
                  "drop_state": load.get("drop", {}).get("location", {}).get("state"),
                  "drop_country": load.get("drop", {}).get("location", {}).get("country"),
                  "drop_latitude": load.get("drop", {}).get("location", {}).get("latitude"),
                  "drop_longitude": load.get("drop", {}).get("location", {}).get("longitude"),
                  "drop_date": load.get("drop", {}).get("startDate"),

                  "equipment_type": equipment_display_names,
                  # "equipment_type_ids": equipment_type_ids,
                  "is_full_load": False if load.get("loadSize", {}).get("fullLoad")=="false" else True,
                  "load_length": load.get("loadSize", {}).get("length"),
                  "load_weight": load.get("loadSize", {}).get("weight"),
                  "load_height": load.get("loadSize", {}).get("height"),
                  "load_width": load.get("loadSize", {}).get("width"),

                  "rate": load.get("rate"),
                  "comment": load.get("comment"),

                  "contact_name": load.get("contactInfo", {}).get("dispatcherName"),
                  "contact_phone": load.get("contactInfo", {}).get("phoneNumber"),
                  "contact_email": load.get("user", {}).get("userName"),
                  "contact_fax": load.get("user", {}).get("fax"),
                  "company_name": load.get("user", {}).get("companyName"),
                  "company_email": load.get("user", {}).get("email"),
                  "MC": next((auth.get("numericValue") for auth in load.get("user", {}).get("authorities", []) if auth.get("type", {}).get("name") == "MC"), None),
                  "DOT": next((auth.get("numericValue") for auth in load.get("user", {}).get("authorities", []) if auth.get("type", {}).get("name") == "DOT"), None),
                  "estimated_distance": load.get("estimatedDistance"),
                  "hash": load.get("hash")
              })

      return extracted_data
      # print(glob.glob('loadReq/*.json'))
  else:
      print('folder does not exists')


# Data ingestion-Truckstop Loadboard

In [34]:
def get_truckstop_data():
  extracted_data = []
  folder_path = '/content/drive/MyDrive/freight_analysis/ts_loadReq/'

  # Check if the folder exists
  if os.path.exists(folder_path):
      # Loop through all JSON files in the specified folder
      for f in glob.glob(os.path.join(folder_path, '*.json')):
          try:
              with open(f, 'r', encoding='utf-8') as file:
                  # The JSON data is an object with a "loads" key
                  data_from_file = json.load(file)
                  loads = data_from_file.get('loads', [])

          except FileNotFoundError:
              print(f"Error: The JSON file '{f}' was not found. Please ensure it's in the correct directory.")
              continue
          except json.JSONDecodeError as e:
              print(f"Error decoding JSON from '{f}': {e}")
              continue
          except Exception as e:
              print(f"An unexpected error occurred with file '{f}': {e}")
              continue

          all_processed_equipment_names = set()
          # Iterate over the list of loads
          for load in loads:
              # Find the pickup and delivery stops from the 'stops' list
              pickup_stop = next((stop for stop in load.get('stops', []) if stop.get('type') == 'Pickup'), None)
              delivery_stop = next((stop for stop in load.get('stops', []) if stop.get('type') == 'Delivery'), None)

              # Safely extract data using .get() for keys that may not exist
              pickup_address = pickup_stop.get("address", {}) if pickup_stop else {}
              delivery_address = delivery_stop.get("address", {}) if delivery_stop else {}
              equipment_info = load.get("equipment", {})
              equipment_display_names = [
                  eq_type
                  for eq_type in list(load.get("equipment", []).get("trailerTypes",[]))
                  if load.get("equipment", []).get("trailerTypes") != [] # Ensure displayName exists
              ]

              # Convert equipment display names to database IDs
              equipment_type_ids = []
              for eq_name in equipment_display_names:
                  eq_id = get_or_create_equipment_id(eq_name, cursor, conn)
                  if eq_id is not None:
                      equipment_type_ids.append(eq_id)
                      all_processed_equipment_names.add(eq_name) # Add to set for testing

              # Build the dictionary for the current load with all original keys
              extracted_data.append({
                  "source": 'ts',
                  "source_id": load.get("id"),
                  "load_reference": load.get("brokerLoadId"), # Not available in new JSON
                  "post_date": load.get("createdAt"),
                  "last_updated": load.get("lastExtractedAt"),

                  "pickup_id": pickup_stop.get("stopIndex") if pickup_stop else None,
                  "pickup_city": pickup_address.get("city").title() if pickup_stop else None,
                  "pickup_state": pickup_address.get("state"),
                  "pickup_country": pickup_address.get("countryIso2"),
                  "pickup_latitude": pickup_stop.get("latitude") if pickup_stop else None,
                  "pickup_longitude": pickup_stop.get("longitude") if pickup_stop else None,
                  "pickup_date": load.get("pickup").get("appointmentStartTime") if pickup_stop else None,

                  "drop_id": delivery_stop.get("stopIndex") if delivery_stop else None,
                  "drop_city": delivery_address.get("city").title()if delivery_stop else None,
                  "drop_state": delivery_address.get("state"),
                  "drop_country": delivery_address.get("countryIso2"),
                  "drop_latitude": delivery_stop.get("latitude") if delivery_stop else None,
                  "drop_longitude": delivery_stop.get("longitude") if delivery_stop else None,
                  "drop_date": load.get("delivery").get("appointmentStartTime") if delivery_stop else None,

                  "equipment_type": equipment_info.get("trailerTypes", []),
                  # "equipment_type_ids":  equipment_type_ids, # Cannot be generated without the original function
                  "is_full_load": None, # Not available in new JSON
                  "load_length": equipment_info.get("length"),
                  "load_weight": load.get("weight"),
                  "load_height": equipment_info.get("height"),
                  "load_width": equipment_info.get("width"),

                  "rate": load.get("price"),
                  'rate_per_mile':  load.get("ratePerMile"),
                  "comment": str(load.get("pickup",'').get("note",''))+', ' + str(load.get("delivery",'').get("note",'')), # Not available in new JSON
                  "contact_name": None, # Not available in new JSON
                  "contact_phone": load.get("bookingPhoneNumber"),
                  "contact_email": load.get("biddingEmail"),
                  "contact_fax": None, # Not available in new JSON
                  "company_name": load.get("broker"), # Broker name is the closest match
                  "company_email": load.get("biddingEmail"), # Closest match
                  "estimated_distance": load.get("distance"),
                  "hash": None # Not available in new JSON
              })

      # For demonstration, let's print the first entry
      if extracted_data:
        print("Successfully extracted data from JSON files.")
        print("First extracted load:")
        # print(json.dumps(extracted_data[0], indent=2))
        print(f"\nTotal loads extracted: {len(extracted_data)}")
        return extracted_data
      else:
        print("No loads were extracted.")
  else:
      print(f"Error: The folder '{folder_path}' does not exist.")


#Data Filters

In [35]:
relevant_columns = ['load_reference','equipment_type','load_weight','load_length','pickup','drop','pickup_zone','drop_zone','rate','estimated_distance','rate_per_mile','pickup_date','company_name','comment']
nextload_duplicate_filters = ['equipment_type','load_weight','load_length','pickup','drop','rate','pickup_date','MC','comment','post_date']
truckstop_duplicate_filters = ['equipment_type','load_weight','load_length','pickup','drop','rate','pickup_date','comment','post_date']

#Enrich data-Nextload & Truckstop Loadboard


*   Assigned corresponding regional zone number(Z0-Z9) to each pickup and drop locations (state)
*   Derived estimated delivery date based on pickup date, estimated trip duration, and daily driving hours for solo drivers.
*   Derived estimated number of daily driving hours left after delivery.



In [36]:
standardize_equipment_map = {
    'StepDeck': 'Step Deck',
    'GooseNeck': 'Removable Gooseneck',
    'Gooseneck': 'Removable Gooseneck',
    'Van': 'Dry Van',
    'BoxTruck': 'Straight Truck',
    'Straight Box': 'Straight Truck',
    'CargoVan': 'Cargo Van'
}

In [52]:
def enrich_nextload_df(nextload_df):
  print(f'before deduplication: {len(nextload_df)}')
  nextload_df = nextload_df.drop_duplicates(subset=['hash'])
  print(f'after dropping hash: {len(nextload_df)}')
  nextload_df = nextload_df.explode('equipment_type')
  nextload_df.dropna(subset=['equipment_type'],axis=0, inplace=True)
  print(f'after explode: {len(nextload_df)}')

  nextload_df["equipment_type"] = nextload_df["equipment_type"].replace(standardize_equipment_map)#.apply(lambda x: x.strip() if x is not None else x)
  nextload_df["pickup"] = nextload_df["pickup_city"].astype(str) + "," + nextload_df["pickup_state"].astype(str) + "," + nextload_df["pickup_country"].astype(str)
  nextload_df["drop"] = nextload_df["drop_city"].astype(str) + "," + nextload_df["drop_state"].astype(str) + "," + nextload_df["drop_country"].astype(str)
  nextload_df["lane"] = nextload_df["pickup"].astype(str) + " - " + nextload_df["drop"].astype(str)
  nextload_df['post_date'] = pd.to_datetime(nextload_df['post_date'], utc=True)
  nextload_df = nextload_df.sort_values(by='post_date', ascending=True).drop_duplicates(subset=nextload_duplicate_filters, keep='last')
  nextload_df['is_repost'] = nextload_df.duplicated(subset='load_reference', keep=False)
  zone_df = pd.read_csv('/content/drive/MyDrive/freight_analysis/zone.csv')
  set_zone = zone_df.set_index('Abbreviation')['Zone']
  nextload_df['drop_zone'] = nextload_df['drop_state'].map(set_zone)
  nextload_df['pickup_zone'] = nextload_df['pickup_state'].map(set_zone)

  nextload_df['rate'] = nextload_df['rate']/100
  nextload_df['rate'] = nextload_df['rate'].astype(int)
  nextload_df['rate_per_mile'] = np.where(
  (nextload_df['rate'] == 0) | (nextload_df['estimated_distance'] == 0),
  0,
  round(nextload_df['rate'] / nextload_df['estimated_distance'], 2)
  )
  print(f'after deduplication: {len(nextload_df)}')
  nextload_df['duration_in_hours'] = nextload_df['estimated_distance']/55
  nextload_df['estimated_distance'] = nextload_df['estimated_distance'].astype(int)
  nextload_df['duration_in_driving_days'] = round(nextload_df['duration_in_hours']/10,1)
  nextload_df['duration_in_driving_days'] = nextload_df['duration_in_driving_days'].fillna(0)
  nextload_df['daily_driving_hours_left'] = nextload_df.apply(lambda row: int(max(math.ceil(row.get('duration_in_driving_days')) - round(row.get('duration_in_driving_days'),1), 0) * 10),axis=1
  )
  # Convert pickup_date column to datetime format
  nextload_df['pickup_date'] = pd.to_datetime(nextload_df['pickup_date'], utc=True)
  nextload_df['drop_date'] = pd.to_datetime(nextload_df['drop_date'], utc=True)

  # Floor duration and convert to timedelta
  nextload_df['estimated_delivery_date'] = nextload_df['pickup_date'] + pd.to_timedelta(nextload_df['duration_in_driving_days'].apply(math.floor), unit='D')
  nextload_df['nextload_pickup_date'] = nextload_df.apply(
      lambda row: row['estimated_delivery_date'] if row['daily_driving_hours_left'] > 2
      else row['estimated_delivery_date'] + pd.to_timedelta(1, unit='D'),
      axis=1
  )
  return nextload_df

In [51]:
def enrich_truckstop_df(truckstop_df):
  print(f'before deduplication: {len(truckstop_df)}')
  truckstop_df = truckstop_df.drop_duplicates(subset=['source_id'])
  print(f'after dropping hash: {len(truckstop_df)}')
  truckstop_df = truckstop_df.explode('equipment_type')
  # truckstop_df.dropna(subset=['equipment_type'],axis=0, inplace=True)
  print(f'after explode: {len(truckstop_df)}')


  truckstop_df["equipment_type"] = truckstop_df["equipment_type"].replace(standardize_equipment_map)#.apply(lambda x: x.strip() if x is not None else x)
  truckstop_df["pickup_country"] = truckstop_df["pickup_country"].map({'US':'USA', 'us':'USA'})#.apply(lambda x: x.strip() if x is not None else x)
  truckstop_df["drop_country"] = truckstop_df["drop_country"].map({'US':'USA', 'us':'USA'})
  truckstop_df["pickup"] = truckstop_df["pickup_city"].astype(str) + "," + truckstop_df["pickup_state"].astype(str) + "," + truckstop_df["pickup_country"].astype(str)
  truckstop_df["drop"] = truckstop_df["drop_city"].astype(str) + "," + truckstop_df["drop_state"].astype(str) + "," + truckstop_df["drop_country"].astype(str)
  truckstop_df["lane"] = truckstop_df["pickup"].astype(str) + " - " + truckstop_df["drop"].astype(str)
  truckstop_df['post_date'] = pd.to_datetime(truckstop_df['post_date'], format='ISO8601').dt.tz_convert('UTC')
  truckstop_df = truckstop_df.sort_values(by='post_date', ascending=True).drop_duplicates(subset=truckstop_duplicate_filters, keep='last')
  truckstop_df['is_repost'] = truckstop_df.duplicated(subset='load_reference', keep=False)
  zone_df = pd.read_csv('/content/drive/MyDrive/freight_analysis/zone.csv')
  set_zone = zone_df.set_index('Abbreviation')['Zone']
  truckstop_df['drop_zone'] = truckstop_df['drop_state'].map(set_zone)
  truckstop_df['pickup_zone'] = truckstop_df['pickup_state'].map(set_zone)

  print(f'after deduplication: {len(truckstop_df)}')

  truckstop_df['duration_in_hours'] = truckstop_df['estimated_distance']/55
  truckstop_df['estimated_distance'] = truckstop_df['estimated_distance'].astype(int)
  truckstop_df['duration_in_driving_days'] = round(truckstop_df['duration_in_hours']/10,1)
  truckstop_df['duration_in_driving_days'] = truckstop_df['duration_in_driving_days'].fillna(0)
  truckstop_df['daily_driving_hours_left'] = truckstop_df.apply(lambda row: int(max(math.ceil(row.get('duration_in_driving_days')) - round(row.get('duration_in_driving_days'),1), 0) * 10),axis=1
  )
  # Convert pickup_date column to datetime format
  truckstop_df['pickup_date'] = pd.to_datetime(truckstop_df['pickup_date']).dt.tz_convert('UTC')
  truckstop_df['drop_date'] = pd.to_datetime(truckstop_df['drop_date']).dt.tz_convert('UTC')

  # Floor duration and convert to timedelta
  truckstop_df['estimated_delivery_date'] = truckstop_df['pickup_date'] + pd.to_timedelta(truckstop_df['duration_in_driving_days'].apply(math.floor), unit='D')
  truckstop_df['nextload_pickup_date'] = truckstop_df.apply(
      lambda row: row['estimated_delivery_date'] if row['daily_driving_hours_left'] > 2
      else row['estimated_delivery_date'] + pd.to_timedelta(1, unit='D'),
      axis=1
  )
  return truckstop_df

## Broker Information

In [39]:
brokerinfo = pd.read_json('/content/drive/MyDrive/freight_analysis/ts_loadReq/broker/brokerinfo.json')


In [40]:
def rename_nextload_brokers(nextload_df, brokerinfo):
  # Remove decimal points from strings and convert to integers safely
  # Assuming nextload_df already has an 'MC' column to merge on.

  # Filter and rename the brokerinfo DataFrame to match columns for merge
  brokerinfo = brokerinfo[brokerinfo['mcNumber'].notna()][['legalName','mcNumber','dotNumber']].rename(
      columns={'mcNumber': 'MC', 'dotNumber': 'DOT', 'legalName': 'legalName'}
  )

  # Merge the dataframes on the 'MC' column, bringing in 'legalName' and 'DOT'
  nextload_df = nextload_df.merge(brokerinfo[['MC', 'legalName']], how='left', on='MC')

  # The original company_name column in nextload_df remains untouched.
  # You might want to update it based on the new legalName column.
  # For example:
  nextload_df['company_name'] = nextload_df['legalName'].fillna(nextload_df['company_name'])

  # You can now drop the 'legalName' column if you no longer need it.
  nextload_df = nextload_df.drop(columns=['legalName'])

  return nextload_df

In [41]:
def rename_truckstop_brokers(truckstop_df, brokerinfo):
  # Remove decimal points from strings and convert to integers safely
  # Filter and rename the brokerinfo DataFrame
  brokerinfo_renamed = brokerinfo[brokerinfo['mcNumber'].notna()][['brokerId','legalName','mcNumber','dotNumber']].rename(
      columns={'brokerId': 'company_name', 'mcNumber': 'MC', 'dotNumber': 'DOT'}
  )

  # Merge with the truckstop_df DataFrame
  truckstop_df = truckstop_df.merge(brokerinfo_renamed, how='left', on='company_name')

  # Replace the original company_name column with the new legalName column
  truckstop_df['company_name'] = truckstop_df['legalName']
  truckstop_df = truckstop_df.drop(columns=['legalName'])

  return truckstop_df

# Merge data from Nextload and Truckstop

In [42]:
def safe_concat(df1, df2):
    # Check if both DataFrames have the same columns (names only)
    missing_in_df2 = set(df1.columns) - set(df2.columns)
    missing_in_df1 = set(df2.columns) - set(df1.columns)

    if set(df1.columns) != set(df2.columns):
        raise ValueError(
            f"❌ Column names do not match.\n"
            f"Missing in df2: {missing_in_df2}\n"
            f"Missing in df1: {missing_in_df1}"
        )
    # Check if column order is the same
    if list(df1.columns) != list(df2.columns):
        print("⚠️ Column order mismatch. Reordering df2 to match df1.")
        df2 = df2[df1.columns]

    # Concatenate safely
    return pd.concat([df1, df2], axis=0, ignore_index=True)


In [16]:
nextload_df_original = pd.DataFrame(get_nextload_data())


In [43]:
truckstop_df_original = pd.DataFrame(get_truckstop_data())

Successfully extracted data from JSON files.
First extracted load:

Total loads extracted: 5964


In [18]:
nextload_df = nextload_df_original.copy()
nextload_df = enrich_nextload_df(nextload_df)
nextload_df = rename_nextload_brokers(nextload_df, brokerinfo)

before deduplication: 78751
after dropping hash: 38215
after explode: 46560
after deduplication: 39234


In [57]:
truckstop_df = truckstop_df_original.copy()
truckstop_df = enrich_truckstop_df(truckstop_df)
truckstop_df = rename_truckstop_brokers(truckstop_df, brokerinfo)

before deduplication: 5964
after dropping hash: 5958
after explode: 7528
after deduplication: 7528


In [20]:
full_df = safe_concat(nextload_df, truckstop_df)
full_df.shape

⚠️ Column order mismatch. Reordering df2 to match df1.


(45559, 49)

In [2]:
# full_df.to_csv('/content/drive/MyDrive/freight_analysis/full_df.csv')
# nextload_df_original.to_csv('/content/drive/MyDrive/freight_analysis/nextload_df_original.csv')
# truckstop_df_original.to_csv('/content/drive/MyDrive/freight_analysis/truckstop_df_original.csv')

In [24]:
unrated_data = full_df[full_df['rate'] < 1]
rated_data = full_df[full_df['rate'] > 0]
agg_rated = rated_data.groupby(['company_name','pickup','drop']).agg({'rate':'mean','rate_per_mile':'mean','pickup_date':'size'}).reset_index()
agg_rated.shape

(10505, 6)

In [25]:
full_df[full_df.duplicated(subset=['load_reference'], keep=False)].sort_values(by='load_reference', ascending=False)

,source,source_id,load_reference,post_date,last_updated,pickup_id,pickup_city,pickup_state,pickup_country,pickup_latitude,pickup_longitude,pickup_date,drop_id,drop_city,drop_state,drop_country,drop_latitude,drop_longitude,drop_date,equipment_type,is_full_load,load_length,load_weight,load_height,load_width,rate,comment,contact_name,contact_phone,contact_email,contact_fax,company_name,company_email,MC,DOT,estimated_distance,hash,pickup,drop,lane,is_repost,drop_zone,pickup_zone,rate_per_mile,duration_in_hours,duration_in_driving_days,daily_driving_hours_left,estimated_delivery_date,nextload_pickup_date
41491,ts,2e2006af-bef3-4f9a-8d22-c1b3e2356972,fc78ef38af8b7a4b68ceffd73c1e8072d58fbbc2,2025-08-12 07:02:59.269000+00:00,2025-08-12T11:33:02.618Z,0.000000e+00,Ontario,CA,USA,34.077968,-117.587449,2025-08-12 07:00:00+00:00,1.000000e+00,Ames,IA,USA,42.065736,-93.694387,2025-08-13 05:00:00+00:00,Dry Van,None,20.0,10000.00,NaN,NaN,NaN,"FROM S EL MONTE CALIF PARTIAL 11 SKIDS 10000, None",None,None,jtrenkamp@onlinefreight.com,None,Online Freight Services,jtrenkamp@onlinefreight.com,315784.0,1563514.0,1685,None,"Ontario,CA,USA","Ames,IA,USA","Ontario,CA,USA - Ames,IA,USA",True,Z5,Z9,NaN,30.646976,3.1,9,2025-08-15 07:00:00+00:00,2025-08-15 07:00:00+00:00
41492,ts,2e2006af-bef3-4f9a-8d22-c1b3e2356972,fc78ef38af8b7a4b68ceffd73c1e8072d58fbbc2,2025-08-12 07:02:59.269000+00:00,2025-08-12T11:33:02.618Z,0.000000e+00,Ontario,CA,USA,34.077968,-117.587449,2025-08-12 07:00:00+00:00,1.000000e+00,Ames,IA,USA,42.065736,-93.694387,2025-08-13 05:00:00+00:00,NaN,None,20.0,10000.00,NaN,NaN,NaN,"FROM S EL MONTE CALIF PARTIAL 11 SKIDS 10000, None",None,None,jtrenkamp@onlinefreight.com,None,Online Freight Services,jtrenkamp@onlinefreight.com,315784.0,1563514.0,1685,None,"Ontario,CA,USA","Ames,IA,USA","Ontario,CA,USA - Ames,IA,USA",True,Z5,Z9,NaN,30.646976,3.1,9,2025-08-15 07:00:00+00:00,2025-08-15 07:00:00+00:00
43978,ts,4071196e-214f-40b9-8f81-40c8dd29f5f9,fc3ceed4-3b4a-4db7-b05e-b33701408cd1,2025-08-12 19:45:08.273000+00:00,2025-08-12T19:45:08.276Z,0.000000e+00,Irving,TX,USA,32.837684,-96.893196,2025-08-12 03:00:00+00:00,1.000000e+00,Jacksonville,FL,USA,30.351634,-81.764696,2025-08-14 00:00:00+00:00,Dry Van,None,NaN,43363.35,NaN,NaN,NaN,"None, None",None,"tel:+1-317-635-6207,,2",None,None,SPOT Freight INC,None,665776.0,2243571.0,996,None,"Irving,TX,USA","Jacksonville,FL,USA","Irving,TX,USA - Jacksonville,FL,USA",True,Z3,Z7,NaN,18.119421,1.8,1,2025-08-13 03:00:00+00:00,2025-08-14 03:00:00+00:00
43977,ts,4071196e-214f-40b9-8f81-40c8dd29f5f9,fc3ceed4-3b4a-4db7-b05e-b33701408cd1,2025-08-12 19:45:08.273000+00:00,2025-08-12T19:45:08.276Z,0.000000e+00,Irving,TX,USA,32.837684,-96.893196,2025-08-12 03:00:00+00:00,1.000000e+00,Jacksonville,FL,USA,30.351634,-81.764696,2025-08-14 00:00:00+00:00,NaN,None,NaN,43363.35,NaN,NaN,NaN,"None, None",None,"tel:+1-317-635-6207,,2",None,None,SPOT Freight INC,None,665776.0,2243571.0,996,None,"Irving,TX,USA","Jacksonville,FL,USA","Irving,TX,USA - Jacksonville,FL,USA",True,Z3,Z7,NaN,18.119421,1.8,1,2025-08-13 03:00:00+00:00,2025-08-14 03:00:00+00:00
44916,ts,ff73792b-38f8-4c40-8000-d20501ea7410,f23efd8972b0c9ca6de619b34a9077bab197d323,2025-08-12 21:44:50.435000+00:00,2025-08-13T02:00:02.282Z,0.000000e+00,Fontana,CA,USA,34.051860,-117.466884,2025-08-13 07:00:00+00:00,1.000000e+00,West Valley City,UT,USA,40.697198,-111.940701,NaT,NaN,None,18.0,12000.00,NaN,NaN,950.0,"Equipment: ; Special Info: Justin-ext 4852; Rep: ext 4852, None",None,"tel:+1-316-530-5111,,4852",jking@kingoffreight.com,None,KING OF Freight LLC,jking@kingoffreight.com,659555.0,2243200.0,650,None,"Fontana,CA,USA","West Valley City,UT,USA","Fontana,CA,USA - West Valley City,UT,USA",True,Z8,Z9,1.460919,11.823191,1.2,8,2025-08-14 07:00:00+00:00,2025-08-14 07:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,next

In [26]:
temp_full = full_df
temp_full[temp_full.duplicated(subset=['load_reference'], keep=False)].sort_values(by='load_reference', ascending=False)

,source,source_id,load_reference,post_date,last_updated,pickup_id,pickup_city,pickup_state,pickup_country,pickup_latitude,pickup_longitude,pickup_date,drop_id,drop_city,drop_state,drop_country,drop_latitude,drop_longitude,drop_date,equipment_type,is_full_load,load_length,load_weight,load_height,load_width,rate,comment,contact_name,contact_phone,contact_email,contact_fax,company_name,company_email,MC,DOT,estimated_distance,hash,pickup,drop,lane,is_repost,drop_zone,pickup_zone,rate_per_mile,duration_in_hours,duration_in_driving_days,daily_driving_hours_left,estimated_delivery_date,nextload_pickup_date
41491,ts,2e2006af-bef3-4f9a-8d22-c1b3e2356972,fc78ef38af8b7a4b68ceffd73c1e8072d58fbbc2,2025-08-12 07:02:59.269000+00:00,2025-08-12T11:33:02.618Z,0.000000e+00,Ontario,CA,USA,34.077968,-117.587449,2025-08-12 07:00:00+00:00,1.000000e+00,Ames,IA,USA,42.065736,-93.694387,2025-08-13 05:00:00+00:00,Dry Van,None,20.0,10000.00,NaN,NaN,NaN,"FROM S EL MONTE CALIF PARTIAL 11 SKIDS 10000, None",None,None,jtrenkamp@onlinefreight.com,None,Online Freight Services,jtrenkamp@onlinefreight.com,315784.0,1563514.0,1685,None,"Ontario,CA,USA","Ames,IA,USA","Ontario,CA,USA - Ames,IA,USA",True,Z5,Z9,NaN,30.646976,3.1,9,2025-08-15 07:00:00+00:00,2025-08-15 07:00:00+00:00
41492,ts,2e2006af-bef3-4f9a-8d22-c1b3e2356972,fc78ef38af8b7a4b68ceffd73c1e8072d58fbbc2,2025-08-12 07:02:59.269000+00:00,2025-08-12T11:33:02.618Z,0.000000e+00,Ontario,CA,USA,34.077968,-117.587449,2025-08-12 07:00:00+00:00,1.000000e+00,Ames,IA,USA,42.065736,-93.694387,2025-08-13 05:00:00+00:00,NaN,None,20.0,10000.00,NaN,NaN,NaN,"FROM S EL MONTE CALIF PARTIAL 11 SKIDS 10000, None",None,None,jtrenkamp@onlinefreight.com,None,Online Freight Services,jtrenkamp@onlinefreight.com,315784.0,1563514.0,1685,None,"Ontario,CA,USA","Ames,IA,USA","Ontario,CA,USA - Ames,IA,USA",True,Z5,Z9,NaN,30.646976,3.1,9,2025-08-15 07:00:00+00:00,2025-08-15 07:00:00+00:00
43978,ts,4071196e-214f-40b9-8f81-40c8dd29f5f9,fc3ceed4-3b4a-4db7-b05e-b33701408cd1,2025-08-12 19:45:08.273000+00:00,2025-08-12T19:45:08.276Z,0.000000e+00,Irving,TX,USA,32.837684,-96.893196,2025-08-12 03:00:00+00:00,1.000000e+00,Jacksonville,FL,USA,30.351634,-81.764696,2025-08-14 00:00:00+00:00,Dry Van,None,NaN,43363.35,NaN,NaN,NaN,"None, None",None,"tel:+1-317-635-6207,,2",None,None,SPOT Freight INC,None,665776.0,2243571.0,996,None,"Irving,TX,USA","Jacksonville,FL,USA","Irving,TX,USA - Jacksonville,FL,USA",True,Z3,Z7,NaN,18.119421,1.8,1,2025-08-13 03:00:00+00:00,2025-08-14 03:00:00+00:00
43977,ts,4071196e-214f-40b9-8f81-40c8dd29f5f9,fc3ceed4-3b4a-4db7-b05e-b33701408cd1,2025-08-12 19:45:08.273000+00:00,2025-08-12T19:45:08.276Z,0.000000e+00,Irving,TX,USA,32.837684,-96.893196,2025-08-12 03:00:00+00:00,1.000000e+00,Jacksonville,FL,USA,30.351634,-81.764696,2025-08-14 00:00:00+00:00,NaN,None,NaN,43363.35,NaN,NaN,NaN,"None, None",None,"tel:+1-317-635-6207,,2",None,None,SPOT Freight INC,None,665776.0,2243571.0,996,None,"Irving,TX,USA","Jacksonville,FL,USA","Irving,TX,USA - Jacksonville,FL,USA",True,Z3,Z7,NaN,18.119421,1.8,1,2025-08-13 03:00:00+00:00,2025-08-14 03:00:00+00:00
44916,ts,ff73792b-38f8-4c40-8000-d20501ea7410,f23efd8972b0c9ca6de619b34a9077bab197d323,2025-08-12 21:44:50.435000+00:00,2025-08-13T02:00:02.282Z,0.000000e+00,Fontana,CA,USA,34.051860,-117.466884,2025-08-13 07:00:00+00:00,1.000000e+00,West Valley City,UT,USA,40.697198,-111.940701,NaT,NaN,None,18.0,12000.00,NaN,NaN,950.0,"Equipment: ; Special Info: Justin-ext 4852; Rep: ext 4852, None",None,"tel:+1-316-530-5111,,4852",jking@kingoffreight.com,None,KING OF Freight LLC,jking@kingoffreight.com,659555.0,2243200.0,650,None,"Fontana,CA,USA","West Valley City,UT,USA","Fontana,CA,USA - West Valley City,UT,USA",True,Z8,Z9,1.460919,11.823191,1.2,8,2025-08-14 07:00:00+00:00,2025-08-14 07:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,next

In [ ]:
temp_full.groupby('load_reference')[temp_full.columns.tolist()].agg(number_of_posts=('load_reference','size'), days_since_original_post=())

#Analysis

## 1. Top N Lanes
compares the top lanes for each equipment type based on average rate for the previous *n* days

In [ ]:
def analyze_top_n_lanes(dataframe: pd.DataFrame, n_days: int = 3, top_n: int = 5) -> pd.DataFrame:
    """
    Analyzes top N lanes based on average rate over the last n business days.
    """
    df = dataframe.copy()

    # Filter for the last n business days and create a copy to avoid the warning
    cutoff_date = pd.to_datetime(date.today(), utc=True) - pd.offsets.BDay(n_days)
    filtered_data = df[(df['post_date'].dt.weekday < 5) & (df['post_date'] >= cutoff_date)].copy()

    # Add date-only column to the independent copy
    filtered_data['date_posted'] = filtered_data['post_date'].dt.date

    # Group and calculate metrics
    lane_metrics = filtered_data.groupby(['date_posted','pickup_state', 'drop_state', 'equipment_type']).agg(
        avg_rate=('rate_per_mile', 'mean'),
        load_count=('post_date', 'size')
    ).reset_index()

    # Sort and return top N lanes
    top_lanes = lane_metrics.sort_values(by='load_count', ascending=False)
    return top_lanes

In [ ]:
top_n_lanes_result = analyze_top_n_lanes(rated_data, n_days=10, top_n=3)

In [ ]:

print("--- Top N Lanes Analysis ---")
# print(top_n_lanes_result.to_string())
print(top_n_lanes_result[(top_n_lanes_result['date_posted']>=(date.today()-timedelta(days=1))) & (top_n_lanes_result['load_count']>2)].sort_values(by='load_count', ascending=False))

print("\n" + "="*50 + "\n")

## weekly trend analysis

compares the *average rate* and *number of loads* posted for lanes on each day of the week over a given number of week(s)

In [ ]:
def analyze_weekly_trends(dataframe: pd.DataFrame, n_weeks: int = 3) -> pd.DataFrame:
    """
    Analyzes week-on-week trends by weekday for rate and load count.

    This updated version ensures every weekday (Monday-Friday) is present for every
    equipment type, filling in 0s for missing data, and returns a DataFrame with
    'equipment_type' and 'weekday' as regular columns for easier plotting.

    This version is more robust to a TypeError by delaying the Categorical conversion.
    """
    # Create a copy to prevent side effects on the original DataFrame
    df = dataframe.copy()

    # Filter for the last n business days
    cutoff_date = pd.to_datetime(date.today(), utc=True) - pd.to_timedelta(n_weeks, unit='W')
    filtered_data = df[(df['post_date'].dt.weekday < 5) & (df['post_date'] >= cutoff_date)]

    # Normalize data by exploding 'equipment_type'
    normalized_data = filtered_data.explode('equipment_type')

    # Add week number and weekday columns directly from the post_date
    normalized_data['week'] = normalized_data['post_date'].dt.isocalendar().week.astype(int)

    # --- CHANGE: Set weekday as a string. Categorical is now handled later. ---
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    normalized_data['weekday'] = normalized_data['post_date'].dt.day_name()

    # Aggregate metrics by equipment_type, weekday, and week directly
    weekly_trends = normalized_data.groupby(['equipment_type', 'weekday', 'week','lane']).agg(
        avg_rate=('rate_per_mile', 'mean'),
        load_count=('rate_per_mile', 'size')
    ).reset_index()

    # Create a DataFrame with all possible combinations of Equipment Type, Weekday, and Week
    all_equipment = normalized_data['equipment_type'].unique()
    all_weeks = normalized_data['week'].unique()
    all_lanes = normalized_data['lane'].unique()

    # --- CHANGE: Use the Categorical type here to ensure correct order in the final output ---
    all_weekdays = pd.Categorical(weekday_order, categories=weekday_order, ordered=True)

    all_combinations = pd.MultiIndex.from_product(
        [all_equipment, all_weekdays, all_weeks, all_lanes],
        names=['equipment_type', 'weekday', 'week','lane']
    ).to_frame(index=False)

    # Merge the calculated trends with the full combinations DataFrame
    merged_trends = all_combinations.merge(
        weekly_trends,
        on=['equipment_type', 'weekday', 'week', 'lane'],
        how='left'
    )

    # Fill NaN values (where there was no data) with 0
    merged_trends = merged_trends.fillna(0)
    merged_trends['avg_rate'] = merged_trends['avg_rate'].round(2)
    # Cast load_count to integer since it's a count
    merged_trends['load_count'] = merged_trends['load_count'].astype(int)

    # Create the pivot table
    # The `observed=False` parameter is added here to ensure all weekday categories are shown,
    # even if no data exists for them in the aggregated data.
    combined_pivot = merged_trends.pivot_table(
        index=['equipment_type', 'weekday', 'lane'],
        columns='week',
        values=['avg_rate', 'load_count'],
        observed=False
    ).fillna(0)
    load_count_cols = [col for col in combined_pivot.columns if col[0] == 'load_count']
    combined_pivot[load_count_cols] = combined_pivot[load_count_cols].astype(int)
    # This sums the load counts across all weeks for each row.
    combined_pivot[('total_load_count', '')] = combined_pivot[load_count_cols].sum(axis=1)
    # Flatten the multi-index to create columns for plotting
    combined_pivot = combined_pivot.reset_index()

    return combined_pivot


In [ ]:
dryvan = rated_data[(rated_data['equipment_type']=='Dry Van') & (rated_data['pickup_state']=='TX')]
weekly_trends_result = analyze_weekly_trends(dryvan, n_weeks=3)


In [ ]:
print("--- Weekly Trends Analysis ---")
print(weekly_trends_result.sort_values(by='total_load_count', ascending=False).head(10).to_string())

# calculate hervesine distance

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on the Earth
    (specified in decimal degrees) using the Haversine formula.
    """
    R = 3958.8  # Earth's radius in miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c
    return distance

# Chain full load trips

In [ ]:

def get_load_chain(data, current, max_depth=1, filter_zones=['Z2', 'Z7', 'Z6', 'Z4', 'Z3']):
    data = data.copy()
    current = current.copy()
    depth = 1

    # # Check if 'pickup_date' is timezone-aware and convert it if necessary
    # if pd.api.types.is_datetime64tz_dtype(data['pickup_date']):
    #     data['pickup_date'] = data['pickup_date'].dt.tz_convert(None)
    # else:
    #     data['pickup_date'] = pd.to_datetime(data['pickup_date'], utc=True).dt.tz_convert(None)

    # # Check if 'drop_date' is timezone-aware and convert it if necessary
    # if pd.api.types.is_datetime64tz_dtype(data['drop_date']):
    #     data['drop_date'] = data['drop_date'].dt.tz_convert(None)
    # else:
    #     data['drop_date'] = pd.to_datetime(data['drop_date'], utc=True).dt.tz_convert(None)

    data['merge_key'] = data['pickup_state'] + '|' + data['pickup_date'].dt.strftime('%m-%d-%Y')
    zone_filtered = data[(data['pickup_zone'].isin(filter_zones)) & (data['drop_zone'].isin(filter_zones))]
    rate_filter = zone_filtered[zone_filtered['rate_per_mile'] > 1.8].copy()
    chained = current.copy()

    while depth <= max_depth:
        previous_depth = depth - 1

        prev_suffix = f'_{previous_depth}' if depth > 1 else ''
        cur_suffix = f'_{depth}'

        if depth == 1:
            drop_date_col = 'drop_date'
            drop_state_col = 'drop_state'
            drop_lat_col = 'drop_latitude'
            drop_lon_col = 'drop_longitude'
        else:
            drop_date_col = f'drop_date{prev_suffix}'
            drop_state_col = f'drop_state{prev_suffix}'
            drop_lat_col = f'drop_latitude{prev_suffix}'
            drop_lon_col = f'drop_longitude{prev_suffix}'

        pickup_lat_col = f'pickup_latitude{cur_suffix}'
        pickup_lon_col = f'pickup_longitude{cur_suffix}'
        deadhead_distance_col = f'deadhead_distance{cur_suffix}'

        # Check for timezone-aware data before converting to string
        if pd.api.types.is_datetime64tz_dtype(chained[drop_date_col]):
            chained['merge_key'] = chained[drop_state_col] + '|' + chained[drop_date_col].dt.strftime('%m-%d-%Y')
        else:
            chained['merge_key'] = chained[drop_state_col] + '|' + chained[drop_date_col].dt.strftime('%m-%d-%Y')

        if drop_date_col not in chained.columns or drop_state_col not in chained.columns:
            print(f"[BREAK] Missing required columns at depth {depth}")
            break

        nextload_matches = chained.merge(
            rate_filter,
            how='left',
            on='merge_key',
            suffixes=('', f'_{depth}')
        )

        nextload_matches.drop(columns=['merge_key'], inplace=True)

        required_cols = [drop_lat_col, drop_lon_col, pickup_lat_col, pickup_lon_col]

        # Use a new variable and .copy() to prevent SettingWithCopyWarning
        nextload_matches_filtered = nextload_matches.dropna(subset=required_cols).copy()

        # Use .loc to assign the new column to prevent the warning
        nextload_matches_filtered.loc[:, deadhead_distance_col] = haversine_distance(
            nextload_matches_filtered[drop_lat_col],
            nextload_matches_filtered[drop_lon_col],
            nextload_matches_filtered[pickup_lat_col],
            nextload_matches_filtered[pickup_lon_col]
        )

        chained = nextload_matches_filtered[nextload_matches_filtered[deadhead_distance_col] < 150].reset_index(drop=True).copy()

        if chained.empty:
            print(f"[BREAK] No valid matches at depth {depth}")
            break

        depth += 1

    return chained

In [ ]:
today = date.today()
dry_van = rated_data[(rated_data['pickup_date'].dt.date >= today) & (rated_data['equipment_type']=='Dry Van')]

# dry_van = curr_loads[curr_loads['equipment_type'].apply( lambda row: 'Dry Van' in  row)]
pickup_texas_filter = dry_van['pickup_state'] == 'TX'
pickup_texas = dry_van[pickup_texas_filter]
pickup_texas = get_load_chain(rated_data, pickup_texas)
pickup_texas['total_duration'] = pickup_texas.filter(like='duration_in_hours').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
pickup_texas['total_distance'] = pickup_texas.filter(like='distance').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True).astype(int)
pickup_texas['total_driving_hours'] = (pickup_texas['total_distance']/55).astype(int)
pickup_texas['total_revenue'] = pickup_texas.filter(like='rate').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
pickup_texas['average_rpm'] = pickup_texas.filter(like='rate_per_mile').apply(pd.to_numeric, errors='coerce').mean(axis=1, skipna=True)
# pickup_texas['total_cost'] = pickup_texas.filter(like='rate').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
pickup_texas['total_cost'] = pickup_texas['total_distance'] * 1.8
pickup_texas['total_profit'] = pickup_texas['total_revenue'] - pickup_texas['total_cost']

pickup_texas.shape

In [ ]:
trip_result = pickup_texas[['total_distance','total_revenue','average_rpm','total_cost','total_profit']]
trip_result.sort_values(by='total_profit', ascending=False)

In [ ]:
profitable_trips = trip_result[trip_result['total_profit'] > 0 ].sort_values(by='total_profit', ascending=False)
print(profitable_trips)

In [ ]:
non_profitable_trips = trip_result[trip_result['total_profit'] < 0 ]
print(non_profitable_trips)

In [ ]:
ftr = pickup_texas[pickup_texas['total_profit']>0].filter(regex='state|city|zone|distance|rate|total|date|hash', axis=1)
ftr#[[relevant_columns]]

In [ ]:
grouped = ftr.groupby('drop_zone_1').agg(
    total_distance=('total_distance', 'mean'),
    total_revenue=('total_revenue', 'mean'),
    total_cost_per_mile=('total_cost', 'mean'),
    total_profit=('total_profit', 'mean')
).rename(
    columns={
        'total_distance': 'average_distance',
        'total_revenue': 'average_revenue',
        'total_cost': 'average_cost_per_trip',
        'total_profit': 'average_profit'
    }
).reset_index()

grouped

# Visualizations

In [ ]:
fig = px.scatter(ftr, x='total_driving_hours', y='total_profit',
                 trendline='ols',
                 title='driving hours by total_profit')
fig.show()

In [ ]:
ftr[(ftr['total_profit']<3000) & (ftr['total_profit']>1800 & (ftr['total_driving_hours'] < 45))]

In [27]:
cursor.execute("SELECT * FROM equipments")
equipment_list = cursor.fetchall()
equipment_list

[(1, 'Dry Van'),
 (2, 'Reefer'),
 (3, 'Flatbed'),
 (4, 'Step Deck'),
 (5, 'Removable Gooseneck'),
 (6, 'Maxi'),
 (7, 'Van'),
 (8, 'StepDeck'),
 (9, 'Conestoga'),
 (10, 'GooseNeck'),
 (11, 'BoxTruck'),
 (12, 'HotShot'),
 (13, 'LessThanTruckload'),
 (14, 'Straight Truck'),
 (15, 'Power Only'),
 (16, 'Straight Box'),
 (17, 'Gooseneck'),
 (18, 'Container'),
 (19, 'Sprinter'),
 (20, 'Auto Carrier'),
 (21, 'Lowboy'),
 (22, 'Double Drop'),
 (23, 'Truck and Trailer'),
 (24, 'CargoVan')]

In [ ]:

def pick_n_drop(data):
# Filter loads with rate_per_mile > 2
  high_value_loads = data[data['rate_per_mile'] > 2]

  # Get distinct pickup and drop states
  distinct_states = numpy.unique(
      numpy.append(
          high_value_loads['pickup_state'].unique(),
          high_value_loads['drop_state'].unique()
      )
  )
  state_df = pd.DataFrame(distinct_states, columns=['state'])

  # Count pickups and drops
  pickup_counts = high_value_loads['pickup_state'].value_counts()
  drop_counts = high_value_loads['drop_state'].value_counts()

  # Combine the counts
  state_summary = pd.concat([pickup_counts, drop_counts], axis=1).fillna(0).astype(int)

  # Rename columns for clarity
  state_summary.columns = ['pickup_total', 'drop_total']

  return state_summary

In [ ]:

rated_data.loc[:,'lead_days'] = (rated_data['pickup_date'] - rated_data['post_date']).dt.days.clip(lower=0)

In [ ]:
# summary of lead days
print(rated_data['lead_days'].describe())

# summary of rate per mile
print(rated_data['rate_per_mile'].describe())

# count shipments by lead_days
lead_counts = rated_data['lead_days'].value_counts().sort_index()
print(lead_counts.head(10))

In [ ]:
fig = px.histogram(rated_data, x='lead_days',
                   nbins=15,
                   title='Distribution of Lead Time (Days)')
fig.show()

In [ ]:
fig = px.scatter(rated_data, x='lead_days', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by Lead Days')
fig.show()

In [ ]:
fig = px.scatter(rated_data, x='estimated_distance', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by estimated distance')
fig.show()

In [ ]:
fig = px.scatter(rated_data[rated_data['load_weight'] < 50000], x='load_weight', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by weight')
fig.show()

In [ ]:

fig = px.histogram(rated_dry_van[rated_dry_van['load_weight'] < 48000], x='load_weight',
                   nbins=15,
                   title='Distribution of Lead Time (Days)')
fig.show()